# 🎵 Spotify Song Recommender
### Content-based recommendation system using audio features

This notebook builds a song recommender that finds similar tracks based on their audio characteristics using cosine similarity.

In [86]:
# Core libraries
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Recommendation
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics.pairwise import cosine_similarity

print("Libraries loaded successfully!")

Libraries loaded successfully!


In [87]:
# Load cleaned dataset
df = pd.read_csv('../data/processed/dataset_clean.csv')
print(f"Dataset shape: {df.shape}")

Dataset shape: (92524, 21)


In [88]:
features = ['danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo']
features_norm = MinMaxScaler().fit_transform(df[features])

In [89]:
# Store normalized features for on-demand similarity calculation
features_norm_df = pd.DataFrame(features_norm, index=df.index, columns=features)
print("Features normalized and ready!")

Features normalized and ready!


In [90]:
# Function to recommend songs based on a given song ID
def recommend_songs(song_id, n=10):
    if song_id not in df['track_id'].values:
        print("Song ID not found in dataset.")
        return []
    
    # Get index of the song
    idx = df.index[df['track_id'] == song_id][0]

    # Compute cosine similarity matrix
    cosine_sim = cosine_similarity(features_norm_df.iloc[[idx]], features_norm_df)
    print(f"Similarity matrix shape: {cosine_sim.shape}")

    # Get similarity scores for the given song
    sim_scores = list(enumerate(cosine_sim[0]))

    # Sort songs based on similarity scores
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    # Get top n similar songs (excluding the input song)
    sim_scores = sim_scores[1:n+1]
    song_indices = [i[0] for i in sim_scores]
    recommended_songs = df.iloc[song_indices][['track_id', 'track_name', 'artists', 'track_genre']]
    recommended_songs.drop_duplicates('track_name', inplace=True)
    return recommended_songs

In [101]:
# Search by song name and artist name, returns top N similar songs

def recommend_by_name(track_name, artist_name, top_n=5):
    # Find the song in the dataset
    song = df[(df['track_name'].str.lower() == track_name.lower()) & (df['artists'].str.lower() == artist_name.lower())]
    
    if song.empty:
        print("Song not found in dataset.")
        return pd.DataFrame()
    
    # Get the track_id of the found song
    song_id = song['track_id'].values[0]
    
    # Get recommendations based on the track_id
    recommendations = recommend_songs(song_id, n=top_n+1)
    
    return recommendations

In [92]:
# Preview some tracks
df[['track_id', 'track_name', 'artists']].head(10)

,track_id,track_name,artists
0,5SuOikwiRyPMVoIQDJUgSV,Comedy,Gen Hoshino
1,4qPNDBW1i3p13qLCt0Ki3A,Ghost - Acoustic,Ben Woodward
2,1iJBSr7s7jYXzM8EGcbK5b,To Begin Again,Ingrid Michaelson;ZAYN
3,6lfxq3CG4xtTiEg7opyCyx,Can't Help Falling In Love,Kina Grannis
4,5vjLSffimiIP26QG5WcN2K,Hold On,Chord Overstreet
5,01MVOl9KtVTNfFiBU9I7dc,Days I Will Remember,Tyrone Wells
6,6Vc5wAMmXdKIAM7WUoEb7N,Say Something,A Great Big World;Christina Aguilera
7,1EzrEOXmMH3G43AXT1y7pA,I'm Yours,Jason Mraz
8,0IktbUcnAGrvD03AWnz3Q8,Lucky,Jason Mraz;Colbie Caillat
9,7k9GuJYLp2AzqokyEdwEw2,Hunger,Ross Copperman


In [93]:
# Trying the recommendation function with a sample song ID
recommend_songs('1iJBSr7s7jYXzM8EGcbK5b')

Similarity matrix shape: (1, 92524)


,track_id,track_name,artists,track_genre
91901,3lqLz5HJ7JFMcOKMNIH3Uo,How Great Is Your Love,Phil Wickham,world-music
79486,05om7Ac9m7wKq1rHn4sHQh,"Monster - From ""Frozen: The Broadway Musical""",Caissie Levy;John Riddle;Original Broadway Cas...,show-tunes
92008,5ELZpvTDGorz9BIE9zaBoZ,I Have This Hope,Tenth Avenue North,world-music
92014,5Rm70bhYNH2eTtHjlwgPY0,The Passion (Live Acoustic) - Bonus,Hillsong Worship;Brooke Ligertwood,world-music
60171,0q8u4fgKU10JDefsWY1nFt,Hoje Eu Sei,Vanessa Da Mata,mpb
49905,12zE0QH17jzC8RHGQaaubz,Thank you,Ryuji Imaichi,j-dance
6732,0tkBOcK7oRVXQJY97zzSvr,Telescope,Cage The Elephant,blues
29483,3Z9wiXDPADfxgovNa2I6ph,Free To Love,Mateo;Arinity;BIMINI,french


In [94]:
# Check the details of the input song
df[df['track_id'] == '1iJBSr7s7jYXzM8EGcbK5b'][['track_name', 'artists', 'track_genre']]

,track_name,artists,track_genre
2,To Begin Again,Ingrid Michaelson;ZAYN,acoustic


In [95]:
# One-Hot Encode track_genre to include genre as a feature
genre_dummies = pd.get_dummies(df['track_genre'])
genre_dummies

,acoustic,afrobeat,alt-rock,alternative,ambient,anime,black-metal,bluegrass,blues,brazil,...,spanish,study,swedish,synth-pop,tango,techno,trance,trip-hop,turkish,world-music
0,True,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
1,True,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2,True,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
3,True,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
4,True,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
92519,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,True
92520,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,True
92521,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,True
92522,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,True


In [96]:
# Concatenate normalized audio features with genre dummies
features_norm_df = pd.concat([features_norm_df, genre_dummies], axis=1)
print("Features with genre dummies added:")
print(features_norm_df.head())

Features with genre dummies added:
   danceability  energy       key  loudness  mode  speechiness  acousticness  \
0      0.686294  0.4610  0.090909  0.791392   0.0     0.148187      0.032329   
1      0.426396  0.1660  0.090909  0.597377   1.0     0.079067      0.927711   
2      0.444670  0.3590  0.000000  0.736123   1.0     0.057720      0.210843   
3      0.270051  0.0596  0.000000  0.573701   1.0     0.037617      0.908635   
4      0.627411  0.4430  0.181818  0.737103   1.0     0.054508      0.470884   

   instrumentalness  liveness   valence  ...  spanish  study  swedish  \
0          0.000001    0.3580  0.718593  ...    False  False    False   
1          0.000006    0.1010  0.268342  ...    False  False    False   
2          0.000000    0.1170  0.120603  ...    False  False    False   
3          0.000071    0.1320  0.143719  ...    False  False    False   
4          0.000000    0.0829  0.167839  ...    False  False    False   

   synth-pop  tango  techno  trance  trip-hop

In [97]:
# Test recommender with the same track_id we used before to see if results change with genre included
recommend_songs('1iJBSr7s7jYXzM8EGcbK5b')

Similarity matrix shape: (1, 92524)


,track_id,track_name,artists,track_genre
43,3TwtrR1yNLY1PMPsrGQpOp,Superman (It's Not Easy),Five For Fighting,acoustic
825,7qLJmyRxLZCf1ifZlJ2gMZ,Over,Johnnyswim,acoustic
40,7LwGBxB0h0CVmkOZxYKn0g,In My Veins - Feat. Erin Mccarley,Andrew Belle,acoustic
490,6fmrddhzAr0StNzWEzlny5,Linda Ronstadt,AJJ,acoustic
875,0jJqIi0uMG8IhGlLx7U85J,Already Home,A Great Big World,acoustic
175,1dcNEEtODRVZEevQ20Cgmy,Nangangamba,Zack Tabudlo,acoustic
626,6b4mWoXrscyajbLOB2qwfG,Drunks,Johnnyswim,acoustic
605,70vp3DhQpLrFQapK0BI5Ug,Running Wild,Jules Larson,acoustic
9,7k9GuJYLp2AzqokyEdwEw2,Hunger,Ross Copperman,acoustic
524,2lvcDndxZGuowLA19g4YDv,Call Me,Gabrielle Aplin,acoustic


In [102]:
# Test recommender by song and artist name
recommend_by_name('Blinding Lights', 'The Weeknd')

Similarity matrix shape: (1, 92524)


,track_id,track_name,artists,track_genre
66427,5HCyWlXZPP0y6Gqq8TgA20,STAY (with Justin Bieber),The Kid LAROI;Justin Bieber,pop
66815,0og9wKFGgFFNQnrBe7eisG,Uff Teri Adaa,Shankar Mahadevan;Alyssa Mendonsa,pop
66388,2eAvDnpXP5W0cVtiI0PUxV,Dandelions,Ruth B.,pop
66963,1D4PL9B8gOg78jiHg3FvBb,Love Story,Taylor Swift,pop
66830,2qgXrzJsry4KgYoJCpuaul,Choo Lo,The Local Train,pop


In [99]:
# Verify multiple songs with same name exist in dataset
df[df['track_name'] == 'Blinding Lights'][['track_name', 'artists', 'track_genre']]

,track_name,artists,track_genre
18757,Blinding Lights,Lucas Estrada;Twan Ray,deep-house
24890,Blinding Lights,Revelries;Victoria Voss,edm
66373,Blinding Lights,The Weeknd,pop


## How the Recommender Works

The recommender uses **cosine similarity** to find songs with similar audio characteristics. Given a `track_id`, it:

1. Normalizes 11 audio features using MinMaxScaler
2. Encodes track genre using One-Hot Encoding
3. Computes cosine similarity between the input song and all 92k tracks
4. Returns the top N most similar songs

### Example
- **Input:** *To Begin Again* — Ingrid Michaelson & ZAYN (acoustic)
- **Output:** 10 acoustic songs with similar energy, danceability and valence

### Search by Name
The `recommend_by_name()` function allows searching by song name and artist, 
making it more user-friendly than searching by `track_id`.

- **Input:** *Blinding Lights* — The Weeknd (pop)
- **Output:** 5 pop songs with similar audio features